# MILK10k Silver validation

This notebook validates and normalizes MILK10k Bronze data into Silver: label normalization, image decode/dimension validation, leakage-control groups, and rejected records.

Most of the actual logic lives in `data_platform.spark_io` as dataset-agnostic shared functions — the same functions ISIC 2019's Silver notebook uses. This notebook is mostly configuration (`data_platform.labels.normalize_diagnosis_labels`, called with `diagnosis_input_columns`/`label_columns` read from `config/bronze/datasets/milk10k.yaml`), reusing the shared pipeline exactly as intended for a second dataset.

Run `05_setup_tables_and_folders.ipynb` and `10_bronze_ingest.ipynb` before this notebook.


In [ ]:
%run ../_setup_env


In [ ]:
from pathlib import Path
import tempfile

from data_platform.files import check_archives_exist, stage_archives_and_extract_metadata
from data_platform.labels import MALIGNANCY_VALUES, normalize_diagnosis_labels
from data_platform.dataset_layout import build_layout, load_dataset_config, load_yaml_config, resolve_archive_paths
from data_platform.spark_io import (
    apply_label_normalization,
    assign_leakage_groups_and_write_inventory,
    build_accepted_rows,
    print_silver_outputs,
    reconcile_bronze_records,
    validate_images,
)

DATASET_CONFIG = load_dataset_config(CONFIG_ROOT / "bronze" / "datasets" / "milk10k.yaml")
STORAGE_CONFIG = load_yaml_config(CONFIG_ROOT / "storage.yaml")
DATASET_CONFIG["landing_root"] = STORAGE_CONFIG["landing_root"]
DATASET_CONFIG["storage_root"] = STORAGE_CONFIG["storage_root"]
layout = build_layout(DATASET_CONFIG)
landing_paths = layout["landing_paths"]
bronze_paths = layout["bronze_paths"]
bronze_tables = layout["bronze_tables"]
silver_tables = layout["silver_tables"]

# The only dataset-specific configuration this notebook needs: the dataset's identity
# on the shared Silver tables, which raw Bronze diagnosis columns feed
# normalize_diagnosis_labels (arity-agnostic, shared with every other dataset), and
# the archive list validate_images streams to decode/validate candidate images -- no
# image bytes are stored anywhere in this project, see
# docs/decisions/004-stream-archives-no-blob-storage.md -- all read from
# config/bronze/datasets/milk10k.yaml, not hardcoded here. Everything else below is
# shared Spark plumbing from data_platform.spark_io.
DATASET_KEY = DATASET_CONFIG["dataset_key"]
DIAGNOSIS_INPUT_COLUMNS = DATASET_CONFIG["diagnosis_input_columns"]
LABEL_COLUMNS = DATASET_CONFIG["label_columns"]
ARCHIVES = resolve_archive_paths(DATASET_CONFIG["archives"], landing_paths, bronze_paths)
LOCAL_ARCHIVE_STAGE_ROOT = Path(tempfile.gettempdir()) / DATASET_CONFIG["dataset_key"] / "archive_stage"

print(f"Bronze source metadata table: {bronze_tables['source_metadata']}")
print(f"Bronze image index table: {bronze_tables['image_index']}")
print(f"Silver image inventory table: {silver_tables['image_inventory']}")
print(f"Silver leakage groups table: {silver_tables['leakage_groups']}")
print(f"Silver rejected records table: {silver_tables['rejected_records']}")


## Stage Archive

`validate_images` streams candidate images directly from their source archive rather than joining Bronze table bytes — no image bytes are stored anywhere in this project (see `docs/decisions/004-stream-archives-no-blob-storage.md`) — so this notebook needs its own locally staged copy of the archive. `check_archives_exist`/`stage_archives_and_extract_metadata` are the same shared helpers `10_bronze_ingest.ipynb` already uses; the metadata re-extraction here is a harmless no-op re-copy (Bronze already wrote it to Bronze metadata) — only the local archive staging matters for this notebook.


In [ ]:
check_archives_exist(ARCHIVES)
stage_archives_and_extract_metadata(ARCHIVES, LOCAL_ARCHIVE_STAGE_ROOT)


## Reconcile Bronze Records

`reconcile_bronze_records` rejects any Bronze image index row with no matching source metadata row before any label or image validation runs. Fully dataset-agnostic, shared via `data_platform.spark_io`.


In [ ]:
source_metadata_df = reconcile_bronze_records(spark, bronze_tables, silver_tables, DATASET_KEY)


## Normalize and Validate Labels

`apply_label_normalization` wraps `data_platform.labels.normalize_diagnosis_labels` as a Spark UDF, applies it to `DIAGNOSIS_INPUT_COLUMNS`, and rejects rows where none of `LABEL_COLUMNS` resolved. It also enforces `data_platform.labels.MALIGNANCY_VALUES` against the `malignancy` column, failing the run if `normalize_diagnosis_labels` ever produces a value outside it — the same enforcement ISIC 2019's notebook uses, now exercised by a second, independent dataset for the first time. `normalize_diagnosis_labels` itself is the exact same shared function ISIC 2019 calls (arity-agnostic); `DIAGNOSIS_INPUT_COLUMNS` (from `config/bronze/datasets/milk10k.yaml`) is the only MILK10k-specific input.


In [ ]:
label_valid_df = apply_label_normalization(
    spark,
    source_metadata_df,
    normalize_diagnosis_labels,
    DIAGNOSIS_INPUT_COLUMNS,
    LABEL_COLUMNS,
    silver_tables,
    DATASET_KEY,
    controlled_vocabularies={"malignancy": MALIGNANCY_VALUES},
)


## Validate Images

`validate_images` streams the staged archive looking only for label-valid candidate images, decodes and validates each one found via `data_platform.validate.decode_batch`, and rejects anything missing from its archive or failing decode/dimension validation. No image bytes are joined from a table or persisted anywhere — each candidate's bytes are read from its source archive, decoded, and discarded (see `docs/decisions/004-stream-archives-no-blob-storage.md`). It also re-verifies each streamed image's checksum against the one Bronze recorded at ingestion, and **raises immediately (fails the whole run) on any mismatch** — source archives must never change after ingestion, and this is what actually checks that rather than just assuming it (see `docs/decisions/005-immutable-source-archives-checksum-verified.md`). Fully dataset-agnostic, shared via `data_platform.spark_io`.


In [ ]:
image_valid_df = validate_images(spark, label_valid_df, ARCHIVES, silver_tables, DATASET_KEY)


## Assign Leakage-Control Groups and Write Image Inventory

`build_accepted_rows` joins label-valid metadata to validated images into one accepted-row DataFrame, tagged with `DATASET_KEY`. MILK10k's Bronze metadata has no `patient_id` column at all (only `lesion_id`) — `build_accepted_rows` already fills it with `NULL` rather than failing, per `docs/data_contract.md`. `assign_leakage_groups_and_write_inventory` groups them (exact duplicates by `source_checksum` first, then `lesion_id`, then `patient_id`, otherwise a singleton — all scoped per dataset) and writes `silver.leakage_groups`/`silver.image_inventory` via idempotent `MERGE INTO`. Both are dataset-agnostic — only `LABEL_COLUMNS` and `DATASET_KEY` vary per dataset.


In [ ]:
accepted_df = build_accepted_rows(label_valid_df, image_valid_df, LABEL_COLUMNS, DATASET_KEY)
image_inventory_df, leakage_groups_df = assign_leakage_groups_and_write_inventory(spark, accepted_df, LABEL_COLUMNS, silver_tables)


## Review Outputs

`print_silver_outputs` shows accepted-vs-rejected summary stats for this run: rejection reasons, validation status, per-label-column distribution, and leakage-group sizes. Fully dataset-agnostic, shared via `data_platform.spark_io`.

In [ ]:
print_silver_outputs(spark, display, silver_tables, LABEL_COLUMNS, DATASET_KEY, image_inventory_df, leakage_groups_df)
